# Search V2 Test Runner

Scratch notebook for exercising the V2 search pipeline one stage at a time on a single query. Currently wired up to run Step 0 (flow routing) in isolation; more stages can be added as they come online.

## Cell 1 — Setup, imports, DB readiness

Run once. Opens the Postgres pool, initializes Redis, and pings Qdrant (idempotent — safe to rerun). Every import used anywhere below lives here.

In [1]:
import sys, time
from datetime import date
from pathlib import Path

# Ensure project root is on the path so imports resolve.
project_root = str(Path.cwd().parent) if Path.cwd().name == "search_v2" else str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from dotenv import load_dotenv
load_dotenv(Path(project_root) / ".env")

from implementation.llms.generic_methods import LLMProvider

# Pipeline steps
from search_v2.step_0 import run_step_0
from search_v2.step_1 import run_step_1
from search_v2.steps_0_2_orchestrator import run_steps_0_to_2

# Shared schemas
from schemas.step_0_flow_routing import Step0Response
from schemas.step_1 import Step1Response
from schemas.step_2 import Step2Response
from schemas.enums import SearchFlow

# DB
from db.postgres import pool as postgres_pool
from db.qdrant import check_qdrant
from db.redis import init_redis, check_redis
import db.redis as _redis_module

# DB readiness (idempotent — mirrors test_stage_1_to_4.ipynb)
if postgres_pool._closed:
    await postgres_pool.open()
    print("Postgres: pool opened")
else:
    print("Postgres: pool already open")

if _redis_module._redis_client is None:
    await init_redis()
    print(f"Redis:    initialized ({await check_redis()})")
else:
    print(f"Redis:    already initialized ({await check_redis()})")

print(f"Qdrant:   {await check_qdrant()}")


/Users/michaelkeohane/Documents/movie-finder-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Postgres: pool opened
Redis:    initialized (ok)
Qdrant:   ok


## Cell 2 — Configuration

Set the query and LLM preset. Uncomment one preset block or assign the three variables manually.

In [3]:
# ---- Query ----
query = "movies my dad would love"

# ---- Presets (uncomment one) ----
# Kimi K2.5 — no thinking
# provider, model, kwargs = LLMProvider.KIMI, "kimi-k2.5", {"enable_thinking": False}

# GPT 5.4 Mini — no thinking
provider, model, kwargs = LLMProvider.OPENAI, "gpt-5.4-mini", {"reasoning_effort": "none", "verbosity": "low"}

# Gemini 3 Flash — no thinking
# provider, model, kwargs = LLMProvider.GEMINI, "gemini-3-flash-preview", {"thinking_config": {"thinking_budget": 0}}

# ---- Shared ----
today = date.today()

print(f"Query:    {query}")
print(f"Provider: {provider.value}")
print(f"Model:    {model}")
print(f"Kwargs:   {kwargs or '(defaults)'}")
print(f"Today:    {today.isoformat()}")


Query:    movies my dad would love
Provider: openai
Model:    gpt-5.4-mini
Kwargs:   {'reasoning_effort': 'none', 'verbosity': 'low'}
Today:    2026-04-24


## Cell 3 — Step 1: Spin Generation (isolated)

Runs the raw query through Step 1's spin generator in isolation. Prints the decomposition (hard commitments / soft interpretations / open dimensions), the original-query label, and the two spins with their levers and queries.

In [16]:
query = "dogshit comedies to watch while high"

start = time.perf_counter()
step_1_response, step_1_in_tok, step_1_out_tok, step_1_call_elapsed = await run_step_1(query)
step_1_elapsed = time.perf_counter() - start

print(f"Step 1 completed in {step_1_elapsed:.2f}s (LLM call: {step_1_call_elapsed:.2f}s)")
print(f"Tokens — input: {step_1_in_tok:,}  output: {step_1_out_tok:,}\n")

print("DECOMPOSITION")
print("-" * 72)
print(f"  hard_commitments     ({len(step_1_response.hard_commitments)}): {step_1_response.hard_commitments}")
print(f"  soft_interpretations ({len(step_1_response.soft_interpretations)}): {step_1_response.soft_interpretations}")
print(f"  open_dimensions      ({len(step_1_response.open_dimensions)}): {step_1_response.open_dimensions}")
print(f"  original_query_label: {step_1_response.original_query_label!r}")
print()

print("SPINS")
print("-" * 72)
for i, spin in enumerate(step_1_response.spins, start=1):
    print(f"  Spin {i} — {spin.ui_label!r}")
    print(f"    branching_opportunity: {spin.branching_opportunity}")
    print(f"    distinctness:          {spin.distinctness}")
    print(f"    query:                 {spin.query!r}")
    print()

print("FULL JSON")
print("-" * 72)
print(step_1_response.model_dump_json(indent=2))


Step 1 completed in 2.76s (LLM call: 2.76s)
Tokens — input: 3,360  output: 373

DECOMPOSITION
------------------------------------------------------------------------
  hard_commitments     (1): ['comedies']
  soft_interpretations (2): ['dogshit', 'watch while high']
  open_dimensions      (2): ['era', 'sub-genre like stoner or parody']
  original_query_label: 'Low-Brow Comedies'

SPINS
------------------------------------------------------------------------
  Spin 1 — "So-Bad-It's-Good Disasters"
    branching_opportunity: Reinterprets 'dogshit' (soft interpretation) as 'so-bad-it's-good' cult failures rather than just generic low-quality movies.
    distinctness:          Focuses on notorious cinematic disasters and bizarre cult flops like 'The Room' or 'Mac and Me' that offer unintentional humor, whereas the original might just return poorly reviewed mainstream comedies.
    query:                 "so-bad-it's-good cult failure comedies to watch while high"

  Spin 2 — 'Trippy Surre

## Cell 4 — Step 2: Query Pre-pass (isolated)

Runs the raw query through Step 2's pre-pass LLM in isolation. Prints the overall intention exploration, every attribute fragment with its nested modifiers (polarity / role markers) and coverage evidence (captured meaning → category → fit → atomic rewrite).

In [4]:
from search_v2.step_2 import run_step_2

query = "movies my dad will love"

start = time.perf_counter()
step_2_response, step_2_in_tok, step_2_out_tok, step_2_call_elapsed = await run_step_2(query)
step_2_elapsed = time.perf_counter() - start

print(f"Step 2 completed in {step_2_elapsed:.2f}s (LLM call: {step_2_call_elapsed:.2f}s)")
print(f"Tokens — input: {step_2_in_tok:,}  output: {step_2_out_tok:,}\n")

print("OVERALL QUERY INTENTION")
print("-" * 72)
print(f"  {step_2_response.overall_query_intention_exploration}")
print()

print(f"REQUIREMENTS ({len(step_2_response.requirements)})")
print("-" * 72)
for i, frag in enumerate(step_2_response.requirements, start=1):
    print(f"  [{i}] {frag.query_text!r}")
    print(f"      description: {frag.description}")
    if frag.modifiers:
        print(f"      modifiers ({len(frag.modifiers)}):")
        for m in frag.modifiers:
            print(f"        - [{m.type.value}] {m.original_text!r} — {m.effect}")
    if frag.coverage_evidence:
        print(f"      coverage_evidence ({len(frag.coverage_evidence)}):")
        for j, ev in enumerate(frag.coverage_evidence, start=1):
            print(f"        {j}. captured_meaning: {ev.captured_meaning}")
            print(f"           category:         {ev.category_name.value}")
            print(f"           fit_quality:      {ev.fit_quality}")
            print(f"           atomic_rewrite:   {ev.atomic_rewrite!r}")
    print()

print("FULL JSON")
print("-" * 72)
print(step_2_response.model_dump_json(indent=2))

Step 2 completed in 4.48s (LLM call: 4.48s)
Tokens — input: 7,750  output: 434

OVERALL QUERY INTENTION
------------------------------------------------------------------------
  The user is seeking movie recommendations tailored for their father, implying a specific co-viewing or gifting occasion. This framing suggests a search for content that aligns with stereotypical 'dad' preferences, likely leaning toward specific genres, tones, or themes associated with that demographic. The query is highly subjective and requires unpacking the 'dad' persona into common cinematic archetypes and viewing goals.

REQUIREMENTS (2)
------------------------------------------------------------------------
  [1] 'my dad'
      description: Identifies the target recipient and viewing persona for the recommendations.
      coverage_evidence (3):
        1. captured_meaning: The request is framed around a specific family member as the primary audience.
           category:         Occasion / self-experienc

## Cell 4 — Steps 0-2 Orchestrator

Runs steps 0 and 1 in parallel against the raw query, then dispatches per the routing decision. The standard flow fans out into 1-3 step-2 branches (3 minus the number of non-standard flows that fired). Exact-title and similarity flows are TODO placeholders today; the orchestrator surfaces whether each WOULD have run.

In [ ]:
orch_result = await run_steps_0_to_2(query)

print(f"Total elapsed: {orch_result.total_elapsed:.2f}s\n")

print("STEP 0")
print("-" * 72)
print(f"  elapsed: {orch_result.step0_elapsed:.2f}s  tokens in/out: {orch_result.step0_input_tokens:,} / {orch_result.step0_output_tokens:,}")
etf = orch_result.step0_response.exact_title_flow_data
sim = orch_result.step0_response.similarity_flow_data
print(f"  exact_title  should_be_searched={etf.should_be_searched}  title={etf.exact_title_to_search!r}")
print(f"  similarity   should_be_searched={sim.should_be_searched}  title={sim.similar_search_title!r}")
print(f"  enable_primary_flow={orch_result.step0_response.enable_primary_flow}  primary_flow={orch_result.step0_response.primary_flow.value}")
print()

print("STEP 1")
print("-" * 72)
if orch_result.step1_response is None:
    print(f"  FAILED: {orch_result.step1_error}")
else:
    s1 = orch_result.step1_response
    print(f"  elapsed: {orch_result.step1_elapsed:.2f}s  tokens in/out: {orch_result.step1_input_tokens:,} / {orch_result.step1_output_tokens:,}")
    print(f"  original_query_label: {s1.original_query_label!r}")
    for i, spin in enumerate(s1.spins, start=1):
        print(f"  spin {i}: {spin.ui_label!r}  query={spin.query!r}")
print()

print("FLOW DISPATCH")
print("-" * 72)
print(f"  exact_title flow would execute: {orch_result.exact_title_flow_executed}  (TODO — not yet implemented)")
print(f"  similarity  flow would execute: {orch_result.similarity_flow_executed}  (TODO — not yet implemented)")
print(f"  step 2 branches planned: {len(orch_result.step2_branches)}")
print()

if orch_result.step2_branches:
    print("STEP 2 BRANCHES")
    print("-" * 72)
    for branch in orch_result.step2_branches:
        print(f"  [{branch.kind}] {branch.ui_label!r}  query={branch.query!r}")
        if branch.error is not None:
            print(f"    FAILED: {branch.error}")
        else:
            print(f"    elapsed: {branch.elapsed:.2f}s  tokens in/out: {branch.input_tokens:,} / {branch.output_tokens:,}")
            print(f"    overall_intention: {branch.response.overall_query_intention_exploration}")
            print(f"    requirements ({len(branch.response.requirements)}):")
            for req in branch.response.requirements:
                mods = ""
                if req.modifiers:
                    mods = "  modifiers=[" + ", ".join(
                        f"{m.type.value}:{m.original_text!r}" for m in req.modifiers
                    ) + "]"
                cats = ", ".join(ev.category_name.value for ev in req.coverage_evidence)
                print(f"      - {req.query_text!r}{mods}  → [{cats}]")
        print()


## Cell 5 — Implicit Expectations (Step 2 -> implicit expectations)

Runs Step 2 on the raw query, then feeds that structured output into the implicit-expectations step. Prints the derived booleans plus one row per Step-2 fragment showing how the new step classified it.

In [6]:
import time

from search_v2.implicit_expectations import run_implicit_expectations
from search_v2.step_2 import run_step_2

query = "horor movies to show at halloween party"

start = time.perf_counter()
step_2_response, step_2_in_tok, step_2_out_tok, step_2_call_elapsed = await run_step_2(query)
implicit_result, implicit_in_tok, implicit_out_tok, implicit_call_elapsed = await run_implicit_expectations(
    query,
    step_2_response,
)
elapsed = time.perf_counter() - start

print(f"Total elapsed: {elapsed:.2f}s")
print(f"Step 2        — elapsed: {step_2_call_elapsed:.2f}s  tokens in/out: {step_2_in_tok:,} / {step_2_out_tok:,}")
print(f"Implicit step — elapsed: {implicit_call_elapsed:.2f}s  tokens in/out: {implicit_in_tok:,} / {implicit_out_tok:,}\n")

print("FINAL RESULT")
print("-" * 72)
print(f"  query_intent_summary:          {implicit_result.query_intent_summary}")
print(f"  explicitly_addresses_quality:  {implicit_result.explicitly_addresses_quality}")
print(f"  explicitly_addresses_notability: {implicit_result.explicitly_addresses_notability}")
print(f"  should_apply_quality_prior:    {implicit_result.should_apply_quality_prior}")
print(f"  should_apply_notability_prior: {implicit_result.should_apply_notability_prior}")
print()

print("EXPLICIT ORDERING AXIS ANALYSIS")
print("-" * 72)
print(implicit_result.explicit_ordering_axis_analysis)
print()

print(f"EXPLICIT SIGNALS ({len(implicit_result.explicit_signals)})")
print("-" * 72)
for i, signal in enumerate(implicit_result.explicit_signals, start=1):
    print(f"  [{i}] {signal.query_span!r}")
    print(f"      normalized_description: {signal.normalized_description}")
    print(f"      explicit_axis:          {signal.explicit_axis}")
    print()

print("FULL JSON")
print("-" * 72)
print(implicit_result.model_dump_json(indent=2))


Total elapsed: 5.44s
Step 2        — elapsed: 3.90s  tokens in/out: 7,753 / 292
Implicit step — elapsed: 1.52s  tokens in/out: 2,320 / 165

FINAL RESULT
------------------------------------------------------------------------
  query_intent_summary:          The user is seeking horror movies suitable for a Halloween party, implying a need for seasonal relevance and a crowd-pleasing social atmosphere.
  explicitly_addresses_quality:  False
  explicitly_addresses_notability: False
  should_apply_quality_prior:    True
  should_apply_notability_prior: True

EXPLICIT ORDERING AXIS ANALYSIS
------------------------------------------------------------------------
No explicit ordering axis such as trending, chronology, or semantic extremeness is present. The 'party' context describes the occasion rather than a specific ranking metric.

EXPLICIT SIGNALS (2)
------------------------------------------------------------------------
  [1] 'horor'
      normalized_description: horror genre
      ex

## Cell 6 — Step 2: save for handler drive-through

Runs step 2 and flattens every `coverage_evidence` entry across every `RequirementFragment` into a single indexed list. The next cell reads from `flat_entries` / `step_2_response` / `step_2_query` (all left in the notebook namespace) so you don't rerun step 2 every time you want to poke a different category.

In [3]:
from search_v2.step_2 import run_step_2

# Edit this and rerun. All downstream cells key off `step_2_query`.
step_2_query = "trending comedies"

start = time.perf_counter()
step_2_response, step_2_in_tok, step_2_out_tok, step_2_call_elapsed = await run_step_2(step_2_query)
step_2_elapsed = time.perf_counter() - start

# Flatten every (fragment, evidence) pair into one indexable list so
# Cell 7 can pick a single atom by integer position. Fragment membership
# is preserved so parent/sibling context can be reconstructed later.
flat_entries = []
for frag_idx, frag in enumerate(step_2_response.requirements):
    for ev_idx, ev in enumerate(frag.coverage_evidence):
        flat_entries.append({
            "index": len(flat_entries),
            "fragment_idx": frag_idx,
            "evidence_idx": ev_idx,
            "fragment_text": frag.query_text,
            "category": ev.category_name,
            "fit_quality": ev.fit_quality,
            "atomic_rewrite": ev.atomic_rewrite,
        })

print(f"Step 2 completed in {step_2_elapsed:.2f}s (LLM call: {step_2_call_elapsed:.2f}s)")
print(f"Tokens \u2014 input: {step_2_in_tok:,}  output: {step_2_out_tok:,}")
print(f"Query:  {step_2_query!r}\n")

print("OVERALL QUERY INTENTION")
print("-" * 72)
print(f"  {step_2_response.overall_query_intention_exploration}\n")

print(f"FLAT CATEGORY INDEX ({len(flat_entries)} entries)")
print("-" * 72)
for e in flat_entries:
    print(f"  [{e['index']:>2}] fragment={e['fragment_text']!r}")
    print(f"       category:    {e['category'].value}  (bucket: {e['category'].bucket.value})")
    print(f"       fit_quality: {e['fit_quality'].value}")
    print(f"       atomic:      {e['atomic_rewrite']!r}")
    print()


Step 2 completed in 2.44s (LLM call: 2.44s)
Tokens — input: 7,747  output: 180
Query:  'trending comedies'

OVERALL QUERY INTENTION
------------------------------------------------------------------------
  The user is looking for comedy films that are currently popular or gaining traction. This combines a broad genre requirement with a live popularity signal.

FLAT CATEGORY INDEX (2 entries)
------------------------------------------------------------------------
  [ 0] fragment='trending'
       category:    Trending  (bucket: single)
       fit_quality: clean
       atomic:      'currently trending or popular'

  [ 1] fragment='comedies'
       category:    Top-level genre  (bucket: mutex)
       fit_quality: clean
       atomic:      'comedy genre'



## Cell 7 — Step 3: run handler on the Nth category

Set `category_index` to the flat index printed by Cell 6 and rerun. The cell mirrors `run_handler`'s internal flow but inlines the LLM call and endpoint executions so the two phases can be timed separately. Short-circuits (TRENDING, no_fit) are preserved.

In [5]:
import asyncio

from db.qdrant import qdrant_client
from implementation.llms.generic_methods import generate_llm_response_async
from schemas.enums import CategoryName, FitQuality
from search_v2.stage_3.category_handlers.handler import (
    TIMEOUT_SECONDS,
    _DOWNRANK,
    _EXCLUSION,
    _INCLUSION,
    _build_endpoint_coroutine,
    _classify_wrapper,
    _extract_fired_endpoints,
)
from search_v2.stage_3.category_handlers.handler_result import HandlerResult
from search_v2.stage_3.category_handlers.prompt_builder import (
    build_system_prompt,
    build_user_message,
)
from search_v2.stage_3.category_handlers.schema_factories import get_output_schema
from search_v2.stage_3.trending_query_execution import execute_trending_query

# ---- Pick which flat entry to feed into step 3 ----
category_index = 1

entry_info = flat_entries[category_index]
parent_fragment = step_2_response.requirements[entry_info["fragment_idx"]]
target_entry = parent_fragment.coverage_evidence[entry_info["evidence_idx"]]
sibling_fragments = [
    f for i, f in enumerate(step_2_response.requirements)
    if i != entry_info["fragment_idx"]
]
category = target_entry.category_name

print(f"RUNNING HANDLER on entry [{category_index}]")
print("-" * 72)
print(f"  fragment:       {parent_fragment.query_text!r}")
print(f"  category:       {category.value}  (bucket: {category.bucket.value})")
print(f"  fit_quality:    {target_entry.fit_quality.value}")
print(f"  atomic_rewrite: {target_entry.atomic_rewrite!r}\n")

llm_elapsed = 0.0
query_elapsed = 0.0
llm_in_tok = llm_out_tok = 0
llm_output = None

# ---- Short-circuits mirror run_handler's gating ----
if category == CategoryName.TRENDING:
    print("TRENDING short-circuit \u2014 LLM skipped.\n")
    start = time.perf_counter()
    trending_result = await execute_trending_query(restrict_to_movie_ids=None)
    query_elapsed = time.perf_counter() - start
    handler_result = HandlerResult(
        inclusion_candidates={sc.movie_id: sc.score for sc in trending_result.scores}
    )
elif target_entry.fit_quality == FitQuality.NO_FIT:
    print("NO_FIT short-circuit \u2014 empty HandlerResult.\n")
    handler_result = HandlerResult()
else:
    # ---- Step 1: build prompts + call LLM (no retry \u2014 surface failures) ----
    system_prompt = build_system_prompt(category)
    user_message = build_user_message(
        raw_query=step_2_query,
        overall_query_intention_exploration=step_2_response.overall_query_intention_exploration,
        target_entry=target_entry,
        parent_fragment=parent_fragment,
        sibling_fragments=sibling_fragments,
    )
    response_format = get_output_schema(category)

    start = time.perf_counter()
    llm_output, llm_in_tok, llm_out_tok = await generate_llm_response_async(
        provider=provider,
        user_prompt=user_message,
        system_prompt=system_prompt,
        response_format=response_format,
        model=model,
    )
    llm_elapsed = time.perf_counter() - start

    print(f"LLM OUTPUT  (tokens in/out: {llm_in_tok:,} / {llm_out_tok:,})")
    print("-" * 72)
    print(llm_output.model_dump_json(indent=2))
    print()

    # ---- Steps 2\u20135: extract fired endpoints, classify, execute in parallel ----
    fired = _extract_fired_endpoints(category, llm_output)
    handler_result = HandlerResult()
    execution_plans: list[tuple] = []
    coroutines: list = []

    for route, wrapper in fired:
        target_bucket = _classify_wrapper(wrapper)
        if target_bucket is None:
            # TRAIT + POSITIVE \u2192 preference spec, deferred to the orchestrator.
            handler_result.preference_specs.append(wrapper)
            continue
        execution_plans.append((route, target_bucket))
        coroutines.append(
            asyncio.wait_for(
                _build_endpoint_coroutine(route, wrapper, qdrant_client=qdrant_client),
                timeout=TIMEOUT_SECONDS,
            )
        )

    if coroutines:
        print(f"EXECUTING {len(coroutines)} endpoint(s) in parallel: "
              f"{[(r.value, b) for r, b in execution_plans]}\n")
        start = time.perf_counter()
        outcomes = await asyncio.gather(*coroutines, return_exceptions=True)
        query_elapsed = time.perf_counter() - start

        for (route, target_bucket), outcome in zip(execution_plans, outcomes):
            if isinstance(outcome, BaseException):
                print(f"  {route.value}: FAILED \u2014 {outcome!r}")
                continue
            if target_bucket == _EXCLUSION:
                for sc in outcome.scores:
                    handler_result.exclusion_ids.add(sc.movie_id)
            else:
                bucket = (
                    handler_result.downrank_candidates
                    if target_bucket == _DOWNRANK
                    else handler_result.inclusion_candidates
                )
                for sc in outcome.scores:
                    bucket[sc.movie_id] = bucket.get(sc.movie_id, 0.0) + sc.score
    else:
        print("No endpoints fired \u2014 nothing to execute.\n")

# ---- Timings ----
print("TIMINGS")
print("-" * 72)
print(f"  LLM call:          {llm_elapsed:.2f}s")
print(f"  Query execution:   {query_elapsed:.2f}s")
print(f"  Total:             {llm_elapsed + query_elapsed:.2f}s\n")

# ---- HandlerResult summary ----
print("HANDLER RESULT")
print("-" * 72)
print(f"  inclusion_candidates: {len(handler_result.inclusion_candidates)} movies")
if handler_result.inclusion_candidates:
    top = sorted(handler_result.inclusion_candidates.items(), key=lambda kv: kv[1], reverse=True)[:20]
    for mid, score in top:
        print(f"    {mid:>10}  {score:.4f}")
print()

print(f"  downrank_candidates:  {len(handler_result.downrank_candidates)} movies")
if handler_result.downrank_candidates:
    top = sorted(handler_result.downrank_candidates.items(), key=lambda kv: kv[1], reverse=True)[:20]
    for mid, score in top:
        print(f"    {mid:>10}  {score:.4f}")
print()

print(f"  exclusion_ids:        {len(handler_result.exclusion_ids)} movies")
if handler_result.exclusion_ids:
    preview = sorted(handler_result.exclusion_ids)[:20]
    print(f"    sample: {preview}")
print()

print(f"  preference_specs:     {len(handler_result.preference_specs)}")
for i, spec in enumerate(handler_result.preference_specs):
    print(f"    [{i}] {type(spec).__name__}  "
          f"match_mode={spec.match_mode.value}  polarity={spec.polarity.value}")


RUNNING HANDLER on entry [1]
------------------------------------------------------------------------
  fragment:       'comedies'
  category:       Top-level genre  (bucket: mutex)
  fit_quality:    clean
  atomic_rewrite: 'comedy genre'



ValueError: Gemini async failed to generate response: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Request contains an invalid argument.', 'status': 'INVALID_ARGUMENT'}}